# Dataloading and preprocessing in Transformers

In [7]:
import torch 
import transformers 
import datasets

from datasets import load_dataset

In [35]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [57]:

data_files = {"Train":'/content/drive/MyDrive/Transformer/drugsComTrain_raw.tsv',"Test":'/content/drive/MyDrive/Transformer/drugsComTest_raw.tsv'}

drug_data = load_dataset("csv",data_files=data_files,delimiter="\t")

In [58]:
drug_sample = drug_data['Train'].shuffle(seed=42).select(range(1000))

In [59]:
drug_sample[:5]

{'Unnamed: 0': [87571, 178045, 80482, 159268, 205477],
 'drugName': ['Naproxen', 'Duloxetine', 'Mobic', 'TriNessa', 'Pristiq'],
 'condition': ['Gout, Acute',
  'ibromyalgia',
  'Inflammatory Conditions',
  'Birth Control',
  'Depression'],
 'review': ['"like the previous person mention, I&#039;m a strong believer of aleve, it works faster for my gout than the prescription meds I take. No more going to the doctor for refills.....Aleve works!"',
  '"I have taken Cymbalta for about a year and a half for fibromyalgia pain. It is great\r\nas a pain reducer and an anti-depressant, however, the side effects outweighed \r\nany benefit I got from it. I had trouble with restlessness, being tired constantly,\r\ndizziness, dry mouth, numbness and tingling in my feet, and horrible sweating. I am\r\nbeing weaned off of it now. Went from 60 mg to 30mg and now to 15 mg. I will be\r\noff completely in about a week. The fibro pain is coming back, but I would rather deal with it than the side effects."',

In [63]:
for col in drug_data.keys():
    assert len(drug_data[col]) == len(drug_data[col].unique("Unnamed: 0"))

In [64]:
drug_data = drug_data.rename_column(
    original_column_name = 'Unnamed: 0',
    new_column_name = 'patient_id'
)
drug_data

DatasetDict({
    Train: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount'],
        num_rows: 161297
    })
    Test: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount'],
        num_rows: 53766
    })
})

In [82]:
def lower_case_conditions(sample):
    return {'condition':sample['condition'].lower()}

drug_data = drug_data.filter(lambda x : x['condition'] is not None)
drug_data = drug_data.map(lower_case_conditions)

Filter:   0%|          | 0/160398 [00:00<?, ? examples/s]

Filter:   0%|          | 0/53471 [00:00<?, ? examples/s]

Map:   0%|          | 0/160398 [00:00<?, ? examples/s]

Map:   0%|          | 0/53471 [00:00<?, ? examples/s]

In [83]:
def compute_review_length(sample):
    return {'review_length': len(sample['review'].split())}

drug_data = drug_data.map(compute_review_length)

Map:   0%|          | 0/160398 [00:00<?, ? examples/s]

Map:   0%|          | 0/53471 [00:00<?, ? examples/s]

In [84]:
drug_data['Train'][0]

{'patient_id': 206461,
 'drugName': 'Valsartan',
 'condition': 'left ventricular dysfunction',
 'review': '"It has no side effect, I take it in combination of Bystolic 5 Mg and Fish Oil"',
 'rating': 9.0,
 'date': 'May 20, 2012',
 'usefulCount': 27,
 'review_length': 17}

In [85]:
drug_data = drug_data.filter(lambda x: x['review_length'] > 30)

Filter:   0%|          | 0/160398 [00:00<?, ? examples/s]

Filter:   0%|          | 0/53471 [00:00<?, ? examples/s]

# Tokenizers in Transformers

In [87]:
from datasets import load_dataset 
raw_dataset = load_dataset("kejian/codesearchnet-python-pep8-v1")

dataset_infos.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/89.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/10.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/180000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/20000 [00:00<?, ? examples/s]

In [108]:
from transformers import AutoTokenizer 
tokenizer = AutoTokenizer.from_pretrained('gpt2')

In [109]:
def get_training_corpus():
    dataset = raw_dataset['train']
    for start_idx in range(0,len(dataset),1000):
        sample = dataset[start_idx:start_idx+1000]
        yield sample['text']

training_corpus = get_training_corpus()

In [110]:
new_tokenizer = tokenizer.train_new_from_iterator(training_corpus,52000)

In [118]:
example = """class LinearLayer():
    def __init__(self, input_size, output_size):
        self.weight = torch.randn(input_size, output_size)
        self.bias = torch.zeros(output_size)

    def __call__(self, x):
        return x @ self.weights + self.bias
    """
x = new_tokenizer.tokenize(example)

In [124]:
new_tokenizer.save_pretrained("/content/drive/MyDrive/Transformer/")

('/content/drive/MyDrive/Transformer/tokenizer_config.json',
 '/content/drive/MyDrive/Transformer/tokenizer.json')